# Fine-tune the voice-agent language classifier in Colab

This notebook loads the prepared train, validation and test splits, audits tokenizer coverage, fine-tunes a sequence classifier, evaluates on the untouched test split, and optionally pushes model artifacts to GitHub using Git LFS.

Important: the existing BERT-Tiny checkpoint is an English baseline and may tokenize Indic scripts poorly. Review the tokenizer audit before training. Never use test data for model selection.


## 1. Install and clone repository


In [ ]:
!pip -q install transformers datasets accelerate scikit-learn pandas safetensors
import pandas as pd, numpy as np, json, os, shutil, subprocess, time
from pathlib import Path
from google.colab import files

REPO_URL = "https://github.com/abhinavnautiyalDS/multilingual-voice-language-classifier.git"
REPO_DIR = Path("/content/multilingual-voice-language-classifier")
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
subprocess.run(["git","clone",REPO_URL,str(REPO_DIR)], check=True)
DATA_DIR = REPO_DIR / "data" / "processed"
TRAIN_PATH, VAL_PATH, TEST_PATH = DATA_DIR/"train.csv.gz", DATA_DIR/"validation.csv.gz", DATA_DIR/"test.csv.gz"
for p in [TRAIN_PATH, VAL_PATH, TEST_PATH]:
    if not p.exists():
        raise FileNotFoundError(str(p) + " missing. Run the dataset-preparation notebook and push splits first.")
train_df = pd.read_csv(TRAIN_PATH, compression="gzip")
val_df = pd.read_csv(VAL_PATH, compression="gzip")
test_df = pd.read_csv(TEST_PATH, compression="gzip")
print("Split sizes:", len(train_df), len(val_df), len(test_df))


## 2. Configure labels and audit tokenizer


In [ ]:
from transformers import AutoTokenizer
MODEL_NAME = "google/bert_uncased_L-2_H-128_A-2"
MAX_LENGTH = 96
SEED = 42
OUTPUT_DIR = Path("/content/bert-tiny-language-classifier")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

LABELS = ["en-IN","hi-IN","pa-IN","bn-IN","gu-IN","kn-IN","ml-IN","mr-IN","te-IN","as-IN","ta-IN","none"]
label2id = {label:i for i,label in enumerate(LABELS)}
id2label = {i:label for label,i in label2id.items()}
for name, frame in [("train",train_df),("validation",val_df),("test",test_df)]:
    if not {"text","label"}.issubset(frame.columns):
        raise ValueError(name + " split needs text and label columns")
    frame["text"] = frame["text"].astype(str)
    frame["label"] = frame["label"].astype(str)
unknown_labels = sorted((set(train_df.label) | set(val_df.label) | set(test_df.label)) - set(LABELS))
if unknown_labels:
    raise ValueError("Unexpected labels: " + str(unknown_labels))

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
def unk_stats(texts, sample_size=2000):
    sample = list(texts[:min(sample_size,len(texts))])
    unk_id = tokenizer.unk_token_id
    total = unk = rows_unk = 0
    for text in sample:
        ids = tokenizer(str(text), truncation=True, max_length=MAX_LENGTH)["input_ids"]
        n = sum(t == unk_id for t in ids) if unk_id is not None else 0
        total += len(ids); unk += n; rows_unk += int(n > 0)
    return {"rows":len(sample),"unk_token_rate":unk/max(1,total),"rows_with_unk_rate":rows_unk/max(1,len(sample))}
coverage = {}
for label in LABELS:
    texts = train_df.loc[train_df.label == label, "text"].tolist()
    if texts:
        coverage[label] = unk_stats(texts)
print(json.dumps(coverage, indent=2, ensure_ascii=False))
print("High unknown-token rates for native Indic scripts are a warning to compare an Indic/multilingual model before training.")\nMAX_ALLOWED_UNK_RATE = 0.05\nALLOW_POOR_TOKENIZER_COVERAGE = False  # Set True only to intentionally run a weak-tokenizer baseline experiment.\nif max((x["unk_token_rate"] for x in coverage.values()), default=0) > MAX_ALLOWED_UNK_RATE and not ALLOW_POOR_TOKENIZER_COVERAGE:\n    raise RuntimeError("Tokenizer UNK rate exceeds 5%. Do not fine-tune this baseline on these data yet. Select an Indic/multilingual tokenizer/model, rerun this audit, and only override the guard for an explicitly labeled baseline experiment.")



## 3. Tokenize datasets and fine-tune


In [ ]:
from datasets import Dataset
from transformers import AutoModelForSequenceClassification, Trainer, TrainingArguments, DataCollatorWithPadding
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix
import torch

def make_dataset(frame):
    data = frame[["text","label"]].copy()
    data["labels"] = data["label"].map(label2id)
    ds = Dataset.from_pandas(data[["text","labels"]], preserve_index=False)
    return ds.map(lambda b: tokenizer(b["text"], truncation=True, max_length=MAX_LENGTH), batched=True, remove_columns=["text"])

train_ds, val_ds, test_ds = make_dataset(train_df), make_dataset(val_df), make_dataset(test_df)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=len(LABELS), label2id=label2id, id2label=id2label
)
collator = DataCollatorWithPadding(tokenizer=tokenizer)

def compute_metrics(eval_pred):
    logits, y = eval_pred
    pred = np.argmax(logits, axis=-1)
    p,r,f,_ = precision_recall_fscore_support(y,pred,average="macro",zero_division=0)
    return {"accuracy":accuracy_score(y,pred),"macro_precision":p,"macro_recall":r,"macro_f1":f}

args = TrainingArguments(
    output_dir=str(OUTPUT_DIR/"checkpoints"),
    learning_rate=3e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    num_train_epochs=4,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    logging_steps=100,
    report_to="none",
    fp16=torch.cuda.is_available(),
    save_total_limit=2,
    seed=SEED
)
trainer = Trainer(model=model,args=args,train_dataset=train_ds,eval_dataset=val_ds,
                  data_collator=collator,compute_metrics=compute_metrics)
trainer.train()
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
(OUTPUT_DIR/"label_map.json").write_text(json.dumps({"label2id":label2id,"id2label":id2label},indent=2),encoding="utf-8")
print("Saved model:", OUTPUT_DIR)


## 4. Evaluate once on the test set


In [ ]:
metrics = trainer.evaluate(test_ds, metric_key_prefix="test")
pred_out = trainer.predict(test_ds, metric_key_prefix="test")
y_true = pred_out.label_ids
y_pred = np.argmax(pred_out.predictions, axis=-1)
report = classification_report(y_true,y_pred,labels=list(range(len(LABELS))),target_names=LABELS,output_dict=True,zero_division=0)
cm = confusion_matrix(y_true,y_pred,labels=list(range(len(LABELS))))
(OUTPUT_DIR/"test_metrics.json").write_text(json.dumps(metrics,indent=2),encoding="utf-8")
(OUTPUT_DIR/"classification_report.json").write_text(json.dumps(report,indent=2),encoding="utf-8")
(OUTPUT_DIR/"confusion_matrix.json").write_text(json.dumps({"labels":LABELS,"matrix":cm.tolist()},indent=2),encoding="utf-8")
preds = test_df.copy()
preds["predicted_label"] = [LABELS[i] for i in y_pred]
preds.to_csv(OUTPUT_DIR/"test_predictions.csv.gz",index=False,compression="gzip")
display(pd.DataFrame(report).T)
print(json.dumps(metrics,indent=2))


## 5. CPU latency and bundle


In [ ]:
cpu_model = AutoModelForSequenceClassification.from_pretrained(OUTPUT_DIR).to("cpu").eval()
cpu_tokenizer = AutoTokenizer.from_pretrained(OUTPUT_DIR)
latencies = []
for text in test_df.text.astype(str).head(300):
    inputs = cpu_tokenizer(text,return_tensors="pt",truncation=True,max_length=MAX_LENGTH)
    start = time.perf_counter()
    with torch.inference_mode():
        cpu_model(**inputs)
    latencies.append((time.perf_counter()-start)*1000)
latency_report = {"cpu_latency_p50_ms":float(np.percentile(latencies,50)),
                  "cpu_latency_p90_ms":float(np.percentile(latencies,90)),
                  "samples":len(latencies)}
(OUTPUT_DIR/"latency_report.json").write_text(json.dumps(latency_report,indent=2),encoding="utf-8")
print(json.dumps(latency_report,indent=2))
zip_path = Path("/content/model_bundle.zip")
import zipfile
with zipfile.ZipFile(zip_path,"w",zipfile.ZIP_DEFLATED) as zf:
    for p in OUTPUT_DIR.rglob("*"):
        if p.is_file() and "checkpoints" not in p.parts:
            zf.write(p,arcname=str(p.relative_to(OUTPUT_DIR)))
files.download(str(zip_path))


## 6. Optional push to GitHub using Colab Secret and Git LFS


In [ ]:
from google.colab import userdata
CONFIRM_MODEL_LICENSES = False
if not CONFIRM_MODEL_LICENSES:
    raise RuntimeError("Review base-model license and repository visibility, then set CONFIRM_MODEL_LICENSES = True.")
token = userdata.get("GITHUB_TOKEN")
if not token:
    raise RuntimeError("Add GITHUB_TOKEN in Colab Secrets with repository Contents read/write.")
env = os.environ.copy()
env["GITHUB_TOKEN"] = token
env["GIT_ASKPASS"] = "/content/git-askpass-model.sh"
env["GIT_TERMINAL_PROMPT"] = "0"
Path(env["GIT_ASKPASS"]).write_text('#!/bin/sh\\ncase "$1" in *Username*) echo x-access-token ;; *Password*) echo "$GITHUB_TOKEN" ;; *) echo "" ;; esac\\n',encoding="utf-8")
Path(env["GIT_ASKPASS"]).chmod(0o700)
subprocess.run(["bash","-lc","apt-get -qq update && apt-get -qq install -y git-lfs && git lfs install"],check=True)
model_repo = Path("/content/model-repo")
if model_repo.exists(): shutil.rmtree(model_repo)
subprocess.run(["git","clone",REPO_URL,str(model_repo)],check=True,env=env)
dest = model_repo/"models"/"bert-tiny-language-classifier"
dest.mkdir(parents=True,exist_ok=True)
for p in OUTPUT_DIR.rglob("*"):
    if p.is_file() and "checkpoints" not in p.parts:
        target = dest/p.relative_to(OUTPUT_DIR)
        target.parent.mkdir(parents=True,exist_ok=True)
        shutil.copy2(p,target)
subprocess.run(["git","-C",str(model_repo),"lfs","track","models/**/*.safetensors","models/**/*.bin"],check=True)
subprocess.run(["git","-C",str(model_repo),"add",".gitattributes","models/bert-tiny-language-classifier"],check=True)
subprocess.run(["git","-C",str(model_repo),"config","user.name","Colab Model Training"],check=True)
subprocess.run(["git","-C",str(model_repo),"config","user.email","colab-model-training@users.noreply.github.com"],check=True)
subprocess.run(["git","-C",str(model_repo),"commit","-m","Add fine-tuned model and evaluation artifacts"],check=True)
subprocess.run(["git","-C",str(model_repo),"push","origin","HEAD"],check=True,env=env)
print("Model pushed to GitHub.")


## Deployment checklist

- Inspect per-label precision, recall, F1 and the confusion matrix.
- Evaluate on held-out real call transcripts not used during training or model selection.
- Compare CPU p50/p90 latency in the actual voice-agent pipeline.
- If tokenizer coverage is poor for Indic scripts, compare a multilingual/Indic tokenizer/model before deployment.
